In [ ]:
%pip install -q edr-xarray matplotlib

# 02 - Indexing and fetching

Use EDR query options and xarray indexing on an explicitly selected collection
and variable.

## 1. Set the server

In [ ]:
import httpx
import xarray as xr

import edr_xarray  # registers engine="edr"

server = "https://edr.example.com"

## 2. List collections

Run this cell, then choose one collection id in the next cell.

In [ ]:
response = httpx.get(f"{server}/collections", timeout=30.0)
response.raise_for_status()
collections = response.json()["collections"]

for collection in collections:
    print(collection["id"], "-", collection.get("title", ""))

## 3. Choose a collection

In [ ]:
collection_id = "EO:EUM:DAT:0398"  # Change this to one of the ids listed above.
collection_url = f"{server}/collections/{collection_id}"
collection_url

## 4. Open with EDR filters

Leave these as `None`, or replace them with values your server accepts.

In [ ]:
bbox = (-3.5, 50.2, -2.1, 51.0)
datetime = "2024-01-01T00:00:00Z/2024-01-02T00:00:00Z"
parameter_names = ["FWI"]

ds = xr.open_dataset(
    collection_url,
    engine="edr",
    bbox=bbox,
    datetime=datetime,
    parameter_names=parameter_names,
)

ds

## 5. List variables and dimensions

In [ ]:
print("sizes:")
print(dict(ds.sizes))
print()
print("data variables:")
print(list(ds.data_vars))

## 6. Choose a variable

In [ ]:
variable = "FWI"  # Change this to one of the data variables listed above.
data = ds[variable]

data

## 7. Build a lazy subset

Change `indexer` to dimensions shown in `data.sizes`. `isel` uses
zero-based positions, so the largest valid integer position is `size - 1`.
This cell does not fetch values.

In [ ]:
print(dict(data.sizes))

indexer = {"t": 0, "y": slice(0, 10), "x": slice(0, 10)}
subset = data.isel(indexer)

subset

## 8. Fetch the subset

In [ ]:
loaded = subset.load()
loaded

## 9. Plot when the subset shape supports it

In [ ]:
plot_data = loaded
while plot_data.ndim > 2:
    plot_data = plot_data.isel({plot_data.dims[0]: 0})

if plot_data.ndim == 2:
    plot_data.plot()
elif plot_data.ndim == 1:
    plot_data.plot.line(marker="o")
else:
    print(plot_data.item())

## 10. Close resources

In [ ]:
ds.close()